In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/bronze/Entities.csv", index_col=0)

Limpieza general

In [3]:
df = df.drop(columns=["property_id","agency","agent","location_id","page_url"])

In [4]:
print(len(df))
df = df.drop_duplicates()
print(len(df))

168446
147957


Limpieza Variables:

Primera vuelta limpieza: 

Variable baths

In [5]:
df["baths"].value_counts()

baths
0      35572
3      29060
2      22449
4      17985
6      15362
5      14067
7       5151
1       4792
8       1970
10       839
9        688
12        10
11         5
13         3
14         3
403        1
Name: count, dtype: int64

In [6]:
df = df[df["baths"] < 7]

In [7]:
len(df)

139287

Variable bedrooms: 

In [8]:
df["bedrooms"].value_counts()

bedrooms
3     38996
2     26764
4     21181
5     19768
0     18135
6      8218
1      3925
7       904
8       631
9       326
10      308
11       80
12       21
14        7
15        7
16        4
28        3
25        3
13        2
18        1
21        1
68        1
27        1
Name: count, dtype: int64

In [9]:
df = df[df["bedrooms"]<7]
len(df)

136987

Variables bedroom - bathroom conjuntas: 

In [10]:
df.groupby("bedrooms")["baths"].value_counts().unstack(fill_value=0)

baths,0,1,2,3,4,5,6
bedrooms,,,,,,,
0,17817,74,89,61,31,27,36
1,530,2890,485,14,3,2,1
2,3679,1542,18415,2948,141,21,18
3,5519,226,2406,23719,6428,654,44
4,2874,40,795,1325,9966,5228,953
5,2532,8,153,572,811,7388,8304
6,1777,7,80,304,362,468,5220


In [11]:
df = df[~((df["bedrooms"] == 0) & (df["baths"] > 0))].reset_index(drop=True)

In [12]:
df = df[~((df["baths"] == 0) & (df["bedrooms"] > 1))].reset_index(drop=True)

In [13]:
n_antes = len(df)
df = df.query("not (bedrooms == 0 and baths == 0)").reset_index(drop=True)
print(f"Filas eliminadas: {n_antes - len(df)}")

Filas eliminadas: 17817


Variable city:

In [32]:
df["city"].value_counts()

city
Lahore        26515
Karachi       16366
Islamabad      7102
Rawalpindi     6226
Faisalabad     1586
Name: count, dtype: int64

Variable price

In [15]:
PKR_TO_EUR_2019 = 1 / 179.61   # ≈ 0.005568, valutafx, 18/07/2019

df["price"] = (df["price"] * PKR_TO_EUR_2019).round(2)

Variable property type

In [31]:
df["property_type"].value_counts()

property_type
House            52011
Flat              3109
Upper Portion     1612
Lower Portion      670
Penthouse          202
Farm House         183
Room                 8
Name: count, dtype: int64

In [17]:
df[df["property_type"] == "Room"]

,property_type,price,location,city,province_name,latitude,longitude,baths,purpose,bedrooms,date_added,Total_Area
156,Room,44.54,Gulshan-e-Iqbal Town,Karachi,Sindh,24.898465,67.062500,1,For Rent,1,7/18/2019,26136.096
419,Room,278.38,DHA Defence,Karachi,Sindh,24.830406,67.053764,3,For Rent,3,7/18/2019,5445.020
675,Room,83.51,G-11,Islamabad,Islamabad Capital,33.663818,72.995260,1,For Rent,1,6/27/2019,4356.016
1278,Room,128.06,DHA Defence,Karachi,Sindh,24.834337,67.055397,1,For Rent,1,7/18/2019,19329.821
1557,Room,111.35,G-11,Islamabad,Islamabad Capital,33.669533,73.000073,0,For Rent,1,7/17/2019,4900.518
...,...,...,...,...,...,...,...,...,...,...,...,...
101255,Room,306.22,DHA Defence,Karachi,Sindh,24.782215,67.074537,1,For Rent,1,7/17/2019,5445.000
101592,Room,33.41,Dhoke Banaras Road,Rawalpindi,Punjab,33.592470,73.014950,0,For Rent,1,7/17/2019,272.251
101718,Room,178.16,DHA Defence,Karachi,Sindh,24.805045,67.064323,1,For Rent,1,7/17/2019,11979.044
101857,Room,445.41,DHA Defence,Karachi,Sindh,24.805045,67.064323,1,For Rent,1,7/17/2019,5445.000


In [18]:
mask = (df["bedrooms"] > 1) & (df["property_type"] == "Room")
print(f"Filas que cambian de Room a Flat: {mask.sum()}")

df.loc[mask, "property_type"] = "Flat"

Filas que cambian de Room a Flat: 64


Variable purpose:

In [19]:
df["purpose"].value_counts()

purpose
For Sale    74047
For Rent    28424
Name: count, dtype: int64

In [20]:
df = df[df["purpose"] == "For Sale"]
df = df.drop(columns=["purpose"]).reset_index(drop=True)

In [21]:
df

,property_type,price,location,city,province_name,latitude,longitude,baths,bedrooms,date_added,Total_Area
0,Flat,55676.19,G-10,Islamabad,Islamabad Capital,33.679890,73.012640,2,2,2/4/2019,1089.004
1,Flat,38416.57,E-11,Islamabad,Islamabad Capital,33.700993,72.971492,3,3,5/4/2019,15246.056
2,House,91865.71,G-15,Islamabad,Islamabad Capital,33.631486,72.926559,6,5,7/17/2019,2178.008
3,House,242191.41,Bani Gala,Islamabad,Islamabad Capital,33.707573,73.151199,4,4,4/5/2019,10890.000
4,House,38973.33,DHA Defence,Islamabad,Islamabad Capital,33.492591,73.301339,3,3,7/10/2019,2178.008
...,...,...,...,...,...,...,...,...,...,...,...
74042,Flat,28951.62,Bahria Town Karachi,Karachi,Sindh,25.013015,67.314401,2,2,7/18/2019,11434.542
74043,Flat,41757.14,Bahria Town Karachi,Karachi,Sindh,25.013015,67.314401,3,3,7/18/2019,2178.008
74044,House,48995.04,Bahria Town Karachi,Karachi,Sindh,25.071948,67.338656,4,3,7/18/2019,2178.008
74045,House,77946.66,Bahria Town Karachi,Karachi,Sindh,25.058062,67.306626,3,3,7/18/2019,2178.008


Variable Total_Area: 

In [22]:
df["Total_Area"] = (df["Total_Area"] * 0.09290304).round(2)

In [23]:
df

,property_type,price,location,city,province_name,latitude,longitude,baths,bedrooms,date_added,Total_Area
0,Flat,55676.19,G-10,Islamabad,Islamabad Capital,33.679890,73.012640,2,2,2/4/2019,101.17
1,Flat,38416.57,E-11,Islamabad,Islamabad Capital,33.700993,72.971492,3,3,5/4/2019,1416.40
2,House,91865.71,G-15,Islamabad,Islamabad Capital,33.631486,72.926559,6,5,7/17/2019,202.34
3,House,242191.41,Bani Gala,Islamabad,Islamabad Capital,33.707573,73.151199,4,4,4/5/2019,1011.71
4,House,38973.33,DHA Defence,Islamabad,Islamabad Capital,33.492591,73.301339,3,3,7/10/2019,202.34
...,...,...,...,...,...,...,...,...,...,...,...
74042,Flat,28951.62,Bahria Town Karachi,Karachi,Sindh,25.013015,67.314401,2,2,7/18/2019,1062.30
74043,Flat,41757.14,Bahria Town Karachi,Karachi,Sindh,25.013015,67.314401,3,3,7/18/2019,202.34
74044,House,48995.04,Bahria Town Karachi,Karachi,Sindh,25.071948,67.338656,4,3,7/18/2019,202.34
74045,House,77946.66,Bahria Town Karachi,Karachi,Sindh,25.058062,67.306626,3,3,7/18/2019,202.34


Segunda vuelta limpieza: 

Variable property_type

In [24]:
df = df[~((df["property_type"] == "Flat") & (df["Total_Area"] > 250))].reset_index(drop=True)

In [34]:
n_antes = len(df)
df = df[~((df["property_type"] == "House") & (df["Total_Area"] > 2500))].reset_index(drop=True)
print(f"Casas eliminadas: {n_antes - len(df)}")

Casas eliminadas: 2080


In [35]:
df

,property_type,price,location,city,province_name,latitude,longitude,baths,bedrooms,date_added,Total_Area
0,Flat,55676.19,G-10,Islamabad,Islamabad Capital,33.679890,73.012640,2,2,2/4/2019,101.17
1,House,91865.71,G-15,Islamabad,Islamabad Capital,33.631486,72.926559,6,5,7/17/2019,202.34
2,House,242191.41,Bani Gala,Islamabad,Islamabad Capital,33.707573,73.151199,4,4,4/5/2019,1011.71
3,House,38973.33,DHA Defence,Islamabad,Islamabad Capital,33.492591,73.301339,3,3,7/10/2019,202.34
4,Penthouse,222704.75,F-11,Islamabad,Islamabad Capital,33.679211,72.988787,5,5,6/27/2019,505.86
...,...,...,...,...,...,...,...,...,...,...,...
55710,House,77946.66,Bahria Town Karachi,Karachi,Sindh,25.005740,67.316385,5,4,7/18/2019,354.10
55711,Flat,41757.14,Bahria Town Karachi,Karachi,Sindh,25.013015,67.314401,3,3,7/18/2019,202.34
55712,House,48995.04,Bahria Town Karachi,Karachi,Sindh,25.071948,67.338656,4,3,7/18/2019,202.34
55713,House,77946.66,Bahria Town Karachi,Karachi,Sindh,25.058062,67.306626,3,3,7/18/2019,202.34
